# 01 - PDF Text Extraction Experiments
Portfolio 01 · Production RAG Service

### Objective
Evaluate and compare PDF extraction libraries (`pdfplumber` vs `pypdf`) on real-world academic and technical documents (`attention_is_all_you_need.pdf`, `rag_survey_paper.pdf`).

### Questions Investigated:
1. Does `pdfplumber` preserve two-column academic layouts accurately?
2. How does `pdfplumber` handle tables compared to `pypdf`?
3. How do we extract page numbers and preserve chunk metadata?
4. How do we detect empty or scanned pages?


In [ ]:
import sys
from pathlib import Path
import pdfplumber
from pypdf import PdfReader

CORPUS_PDF = Path("../corpus/pdfs/attention_is_all_you_need.pdf")
print(f"Testing with: {CORPUS_PDF.name} (Exists: {CORPUS_PDF.exists()})")


### 1. Extract Page 1 with pdfplumber


In [ ]:
with pdfplumber.open(CORPUS_PDF) as pdf:
    print(f"Total pages: {len(pdf.pages)}")
    page_1 = pdf.pages[0]
    text_plumber = page_1.extract_text() or ""
    print(f"Page 1 character count (pdfplumber): {len(text_plumber)}")
    print("\n--- Sample text (first 400 chars) ---")
    print(text_plumber[:400])


### 2. Compare with pypdf extraction


In [ ]:
reader = PdfReader(CORPUS_PDF)
pypdf_page_1 = reader.pages[0].extract_text() or ""
print(f"Page 1 character count (pypdf): {len(pypdf_page_1)}")
print(f"Difference: {abs(len(text_plumber) - len(pypdf_page_1))} chars")


### 3. Table Extraction Experiment
Check if the paper has tables on subsequent pages (e.g. Page 8 contains model BLEU scores).


In [ ]:
with pdfplumber.open(CORPUS_PDF) as pdf:
    for idx, page in enumerate(pdf.pages):
        tables = page.extract_tables()
        if tables:
            print(f"Page {idx + 1} contains {len(tables)} table(s)")
            for t_idx, table in enumerate(tables):
                print(f"  Table {t_idx + 1} row count: {len(table)}")
                if len(table) > 0:
                    print(f"  Headers: {table[0][:4]}")
            break


### 4. Design Decision Summary
- **Primary Library Selected:** `pdfplumber` because it provides precise bounding box detection and extracts tables cleanly.
- **Fallback Library:** `pypdf` is retained as a lightweight fallback for fast plain-text extraction if pdfplumber encounters malformed fonts.
